# Investigation 10: t-SNE Projection of Rules x FOVs

This notebook applies **t-SNE** (t-Distributed Stochastic Neighbor Embedding) to the spatial rule data.

Unlike PCA, which forces data into rigid linear axes (often resulting in scattered plots for highly sparse, complex biological data), t-SNE is a non-linear manifold technique. It excels at grouping similar FOVs together into local "islands" or clusters based on their shared spatial rule profiles.

We use this to visualize if FOVs naturally separate by Organ or Pathological Score based purely on their spatial architectures.

## 1. Imports & Configuration
We set up our environment and define the key parameters for the t-SNE algorithm.

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import os
import ast
import warnings

warnings.filterwarnings('ignore')

# --- Global Parameters ---
METRIC_TO_USE = 'Lift'
NO_SELF = True
MIN_FOV_PERCENTAGE = 0.0  # Drop rules appearing in fewer than 10% of FOVs to remove noise

# t-SNE Parameters
TSNE_PERPLEXITY = 30       # Controls the balance between local and global attention
TSNE_RANDOM_STATE = 42     # Ensures the plot looks the same every time we run it

# Paths
RESULTS_PATH = '../../results/full_run/weighted_fpgrowth/data/results_CN.csv'
DATA_DIR = '../../data/MIBIGutCsv'


## 2. Data Loading & Metadata Formatting
We load the rules and attach the clinical metadata (Organ, Pathological score) so we can color our t-SNE plots.

In [7]:
def load_and_format_data():
    print("Loading results and metadata...")
    df_results = pd.read_csv(RESULTS_PATH)
    df_fovs = pd.read_csv(os.path.join(DATA_DIR, 'fovs_metadata.csv'))
    df_biopsy = pd.read_csv(os.path.join(DATA_DIR, 'biopsy_metadata.csv'))
    
    # Merge metadata
    df_fovs = pd.merge(
        df_fovs,
        df_biopsy[['Biopsy_ID', 'Pathological score', 'Clinical score', 'Localization']],
        left_on='Patient',
        right_on='Biopsy_ID',
        how='left'
    )
    
    # Format Organ Labels
    def get_organ(row):
        if pd.notna(row.get("Localization")): return row["Localization"]
        cohort = str(row.get("Cohort", ""))
        if "Colon" in cohort: return "Colon"
        if "Duodenum" in cohort: return "Duodenum"
        return "Unknown"
        
    # Format Clinical Labels consistently
    def get_clean_score(row, score_col):
        if pd.notna(row[score_col]): return str(row[score_col])
        if str(row['FOV']).startswith('S_'): return 'Control_S'
        return 'Control'

    df_fovs["Organ"] = df_fovs.apply(get_organ, axis=1)
    df_fovs["Pathological score"] = df_fovs.apply(lambda row: get_clean_score(row, "Pathological score"), axis=1)
    df_fovs["Clinical score"] = df_fovs.apply(lambda row: get_clean_score(row, "Clinical score"), axis=1)
    
    return df_results, df_fovs

df_results, df_fovs = load_and_format_data()


Loading results and metadata...


## 3. Matrix Construction & Mathematical Preprocessing
To ensure t-SNE doesn't scatter randomly, we apply three strict preprocessing rules:
1. **Frequency Filter**: Drop ultra-rare rules so the algorithm focuses on systemic biology.
2. **Symmetric Transform**: Use `log2` to make Lift attractions and avoidances mathematically symmetric.
3. **Standardization**: Force all rules to an equal variance so loud structural rules don't drown out subtle immune signals.

In [8]:
def build_preprocessed_matrix(df_results):
    print("Cleaning rule strings...")
    def clean_items(item_list_str):
        try:
            items = ast.literal_eval(item_list_str)
            cleaned = [item.replace('_CENTER', '').replace('_NEIGHBOR', '') for item in items]
            return ", ".join(sorted(cleaned))
        except Exception:
            return item_list_str

    df_results['Ant_Clean'] = df_results['Antecedents'].apply(clean_items)
    df_results['Con_Clean'] = df_results['Consequents'].apply(clean_items)
    df_results['Clean_Rule'] = df_results['Ant_Clean'] + " -> " + df_results['Con_Clean']

    if NO_SELF:
        def has_overlap(row):
            ant_clean = set(row['Ant_Clean'].split(', '))
            con_clean = set(row['Con_Clean'].split(', '))
            return not ant_clean.isdisjoint(con_clean)
        overlap_mask = df_results.apply(has_overlap, axis=1)
        df_results = df_results[~overlap_mask].reset_index(drop=True)

    print("Pivoting matrix...")
    df_pivot = df_results.drop_duplicates(subset=['FOV', 'Clean_Rule']) 
    feature_matrix = df_pivot.pivot(index='FOV', columns='Clean_Rule', values=METRIC_TO_USE)
    feature_matrix = feature_matrix.fillna(1.0) # 1.0 is neutral Lift

    print("Filtering rare rules to reduce noise...")
    rule_counts = (feature_matrix != 1.0).sum(axis=0)
    valid_rules = rule_counts[rule_counts >= len(feature_matrix) * MIN_FOV_PERCENTAGE].index
    feature_matrix = feature_matrix[valid_rules]

    print("Applying log2 transform and StandardScaler...")
    # Log2 stabilizes the massive Lift outliers
    feature_matrix_log = np.log2(feature_matrix + 1e-9)
    
    # StandardScaler prevents a few high-variance rules from dominating the plot
    scaled_data = StandardScaler().fit_transform(feature_matrix_log)
    final_matrix = pd.DataFrame(scaled_data, index=feature_matrix.index, columns=feature_matrix.columns)
    
    print(f"Final Matrix ready. Shape: {final_matrix.shape}")
    return final_matrix

final_matrix = build_preprocessed_matrix(df_results)


Cleaning rule strings...
Pivoting matrix...
Filtering rare rules to reduce noise...
Applying log2 transform and StandardScaler...
Final Matrix ready. Shape: (288, 587)


## 4. t-SNE Projection
We calculate the 2D t-SNE coordinates for every FOV.

In [9]:
def calculate_tsne(feature_matrix, df_fovs):
    print(f"Running t-SNE (Perplexity: {TSNE_PERPLEXITY})...")
    
    tsne_model = TSNE(
        n_components=2, 
        perplexity=TSNE_PERPLEXITY, 
        random_state=TSNE_RANDOM_STATE
    )
    
    tsne_coordinates = tsne_model.fit_transform(feature_matrix)
    
    # Combine coordinates with FOV metadata for easy plotting
    df_tsne = pd.DataFrame(tsne_coordinates, index=feature_matrix.index, columns=['tSNE_1', 'tSNE_2'])
    df_tsne_merged = df_tsne.reset_index().merge(
        df_fovs[['FOV', 'Organ', 'Clinical score', 'Pathological score']], 
        on='FOV', 
        how='left'
    )
    
    # Fill any empty organs to prevent visualization errors
    df_tsne_merged['Organ'] = df_tsne_merged['Organ'].fillna('Unknown')
    
    return df_tsne_merged

df_tsne_merged = calculate_tsne(final_matrix, df_fovs)


Running t-SNE (Perplexity: 30)...


## 5. Visualizations
We map the t-SNE coordinates using clean, interactive Plotly scatter plots, strictly enforcing intuitive color palettes and categorical ordering.

In [10]:
def visualize_tsne(df_tsne_merged, color_by='Pathological score'):
    color_discrete_map = None
    category_orders = None
    
    # Enforce strict colors and order for clinical scores
    if color_by in ['Pathological score', 'Clinical score']:
        color_discrete_map = {
            'Control_S': '#2E7D32', # Dark Green
            'Control': '#81C784',   # Light Green
            'Mild': '#FFA726',      # Orange
            'Severe': '#D32F2F'     # Red
        }
        category_orders = {color_by: ['Control_S', 'Control', 'Mild', 'Severe']}

    # Build Plot
    fig = px.scatter(
        df_tsne_merged, 
        x='tSNE_1', 
        y='tSNE_2', 
        color=color_by,
        color_discrete_map=color_discrete_map,
        category_orders=category_orders,
        hover_name='FOV',
        title=f"t-SNE of FOVs based on Spatial Rules ({METRIC_TO_USE})",
        subtitle=f"Colored by: {color_by}",
        template='plotly_white',
        width=900,
        height=600
    )

    # Styling
    fig.update_traces(marker=dict(size=10, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')))
    fig.show()



In [11]:
# Display the plots
visualize_tsne(df_tsne_merged, color_by='Organ')
visualize_tsne(df_tsne_merged, color_by='Pathological score')


In [12]:

METRIC_TO_USE = 'Conviction'
NO_SELF = True
MIN_FOV_PERCENTAGE = 0.0  # Drop rules appearing in fewer than 10% of FOVs to remove noise


visualize_tsne(df_tsne_merged, color_by='Organ')
visualize_tsne(df_tsne_merged, color_by='Pathological score')


In [13]:

METRIC_TO_USE = 'Lift'
NO_SELF = True
MIN_FOV_PERCENTAGE = 0.20  # Drop rules appearing in fewer than 10% of FOVs to remove noise


visualize_tsne(df_tsne_merged, color_by='Organ')
visualize_tsne(df_tsne_merged, color_by='Pathological score')


In [ ]:

METRIC_TO_USE = 'Conviction'
NO_SELF = True
MIN_FOV_PERCENTAGE = 0.20  # Drop rules appearing in fewer than 10% of FOVs to remove noise


visualize_tsne(df_tsne_merged, color_by='Organ')
visualize_tsne(df_tsne_merged, color_by='Pathological score')


: 